In [1]:
from src.preprocessing.io.feature_loader import FeatureLoader
from src.common.constants import Constants as consts
import matplotlib.pyplot as plt

feature_loader = FeatureLoader(feat_suffix=consts.wavlm_emb_suffix)
meta, feat = feature_loader.load_data()
meta, feat = feature_loader.sample_data(meta, feat, fraction=0.1)
print(meta.shape, feat.shape)

2026-05-20 00:29:02 | INFO     | FeatureLoader | Using WavLM embeddings suffix
2026-05-20 00:29:02 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted_wavlm.npy
2026-05-20 00:29:02 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-05-20 00:29:02 | INFO     | FeatureLoader | Loading features from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-05-20 00:29:02 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv


(112588, 9) (112588, 768)


In [2]:
meta = meta[meta["anomaly"] == 0]
feat = feat[meta.index]
print(meta.shape, feat.shape)

(104540, 9) (104540, 768)


In [3]:
from umap import UMAP

umap_model = UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    verbose=True,
)

X_umap_3d = umap_model.fit_transform(feat)

UMAP(angular_rp_forest=True, metric='cosine', n_components=3, verbose=True)
Wed May 20 00:31:20 2026 Construct fuzzy simplicial set
Wed May 20 00:31:20 2026 Finding Nearest Neighbors
Wed May 20 00:31:20 2026 Building RP forest with 21 trees
Wed May 20 00:31:24 2026 NN descent for 17 iterations
	 1  /  17
	 2  /  17
	 3  /  17
	 4  /  17
	 5  /  17
	 6  /  17
	Stopping threshold met -- exiting after 6 iterations
Wed May 20 00:31:29 2026 Finished Nearest Neighbor Search
Wed May 20 00:31:30 2026 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Wed May 20 00:31:39 2026 Finished embedding


In [8]:
mask = meta["config"] == "mls-bonafide"
print(mask.sum())

5056


In [16]:
import numpy as np
import plotly.graph_objects as go

mask_np = mask.to_numpy()
x = X_umap_3d[:, 0]
y = X_umap_3d[:, 1]
z = X_umap_3d[:, 2]

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x[~mask_np],
        y=y[~mask_np],
        z=z[~mask_np],
        mode="markers",
        marker=dict(size=1, opacity=0.3),
        name="mask=0",
    )
 )

fig.add_trace(
    go.Scatter3d(
        x=x[mask_np],
        y=y[mask_np],
        z=z[mask_np],
        mode="markers",
        marker=dict(size=1, opacity=0.9),
        name="mask=1",
    )
 )

fig.update_layout(
    title="UMAP Projection of WavLM Embeddings (3D)",
    width=800,
    height=800,
 )

fig.show()